# Korean Government Bond (국고채) YTM Anomaly Detection

**Purpose**: Detect anomalies in Korean Treasury Bond (KTB) yield-to-maturity data using  
IQR-based rolling detection, maturity-scaled daily change flags, duration/convexity analytics,  
and scipy.optimize.brentq for YTM numerical solving.

**Data source**: Bank of Korea ECOS Open API — Statistical Table 817Y002 (국고채수익률)  
**Maturities**: 1Y / 3Y / 5Y / 10Y  
**Period**: 2022-01-01 ~ 2025-12-31

---
**Relevance to bond evaluation practice**:  
KIS자산평가 채권평가본부는 매일 수천 종목의 채권 수익률을 검증합니다.  
이 노트북은 그 핵심인 '데이터 정합성 검증 → 이상치 탐지 → 수익률 역산' 파이프라인을 구현합니다.

## Section 1 — Imports & Configuration

In [ ]:
import os
import json
import warnings
from io import StringIO

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.optimize import brentq

warnings.filterwarnings('ignore')

# ── Configuration ────────────────────────────────────────────────────────────
ECOS_API_KEY = os.getenv('ECOS_API_KEY', 'sample')   # set env var or use sample mode
START_DATE   = '20220101'
END_DATE     = '20251231'

# ECOS statistical table 817Y002: 국고채수익률 (일별)
STAT_CODE = '817Y002'
MATURITIES = {
    '1Y':  '010190000',
    '3Y':  '010200000',
    '5Y':  '010210000',
    '10Y': '010220000',
}

# Maturity-scaled daily change thresholds (bp)
# Rationale: longer-maturity bonds embed a larger term premium and liquidity premium,
# resulting in naturally wider daily price swings.
BP_THRESHOLDS = {'1Y': 0.03, '3Y': 0.05, '5Y': 0.07, '10Y': 0.10}  # expressed as % (1bp = 0.01%)

print(f"ECOS API mode: {'SAMPLE (no real API call)' if ECOS_API_KEY == 'sample' else 'LIVE'}")
print(f"Period: {START_DATE} ~ {END_DATE}")

## Section 2 — Data Collection (ECOS API)

ECOS Open API endpoint format:  
`/StatisticSearch/{KEY}/json/kr/{START_ROW}/{END_ROW}/{STAT_CODE}/{CYCLE}/{DATE_FROM}/{DATE_TO}/{ITEM_CODE}`

> **Note on pagination**: 3 years of daily data ≈ 750 rows per maturity.  
> We use `END_ROW=2000` to avoid the 100-row default truncation.

In [ ]:
# ── Sample data embedded for reproducibility (no API key needed) ─────────────
SAMPLE_CSV = """date,ytm_1y,ytm_3y,ytm_5y,ytm_10y
2022-01-03,1.39,2.11,2.27,2.44
2022-01-04,1.41,2.14,2.30,2.47
2022-01-05,1.44,2.19,2.35,2.52
2022-01-06,1.46,2.22,2.38,2.55
2022-01-07,1.48,2.25,2.41,2.57
2022-01-10,1.50,2.28,2.44,2.60
2022-01-11,1.51,2.29,2.45,2.61
2022-01-12,1.53,2.31,2.47,2.62
2022-01-13,1.55,2.34,2.50,2.65
2022-01-14,1.57,2.37,2.53,2.68
2022-01-17,1.56,2.35,2.51,2.66
2022-01-18,1.58,2.38,2.54,2.70
2022-01-19,1.61,2.42,2.58,2.74
2022-01-20,1.63,2.45,2.61,2.77
2022-01-21,1.65,2.47,2.63,2.79
2022-01-24,1.67,2.50,2.66,2.82
2022-01-25,1.69,2.52,2.68,2.84
2022-01-26,1.71,2.55,2.71,2.87
2022-01-27,1.72,2.57,2.73,2.89
2022-01-28,1.74,2.59,2.75,2.91
2022-02-07,1.76,2.62,2.78,2.94
2022-02-08,1.78,2.65,2.81,2.97
2022-02-09,1.80,2.67,2.83,2.99
2022-02-10,1.82,2.70,2.86,3.02
2022-02-11,1.84,2.73,2.89,3.05
2022-02-14,1.86,2.76,2.92,3.08
2022-02-15,1.88,2.79,2.95,3.11
2022-02-16,1.90,2.82,2.98,3.14
2022-02-17,1.92,2.85,3.01,3.17
2022-02-18,1.94,2.88,3.04,3.20
2022-03-01,1.97,2.92,3.08,3.24
2022-03-02,1.99,2.95,3.11,3.27
2022-03-03,2.01,2.98,3.14,3.30
2022-03-04,2.03,3.01,3.17,3.33
2022-03-07,2.05,3.04,3.20,3.36
2022-03-08,2.07,3.07,3.23,3.39
2022-03-09,2.10,3.11,3.27,3.43
2022-03-10,2.12,3.14,3.30,3.46
2022-03-11,2.14,3.17,3.33,3.49
2022-03-14,2.16,3.20,3.36,3.52
2022-04-01,2.20,3.25,3.41,3.57
2022-04-04,2.22,3.28,3.44,3.60
2022-04-05,2.25,3.32,3.48,3.64
2022-04-06,2.27,3.35,3.51,3.67
2022-04-07,2.30,3.39,3.55,3.71
2022-04-08,2.32,3.42,3.58,3.74
2022-04-11,2.35,3.46,3.62,3.78
2022-04-12,2.38,3.50,3.66,3.82
2022-04-13,2.40,3.53,3.69,3.85
2022-04-14,2.43,3.57,3.73,3.89
2022-05-02,2.50,3.65,3.81,3.97
2022-05-03,2.53,3.69,3.85,4.01
2022-05-04,2.56,3.73,3.89,4.05
2022-05-09,2.60,3.78,3.94,4.10
2022-05-10,2.63,3.82,3.98,4.14
2022-05-11,2.66,3.86,4.02,4.18
2022-05-12,2.35,3.50,3.68,3.85
2022-05-13,2.38,3.53,3.71,3.88
2022-05-16,2.41,3.57,3.75,3.92
2022-05-17,2.44,3.61,3.79,3.96
2022-06-01,2.90,3.85,3.98,4.12
2022-06-02,2.93,3.89,4.02,4.16
2022-06-03,2.96,3.93,4.06,4.20
2022-06-07,3.00,3.98,4.11,4.25
2022-06-08,3.03,4.02,4.15,4.29
2022-06-09,3.40,4.20,4.32,4.45
2022-06-10,3.43,4.24,4.36,4.49
2022-06-13,3.46,4.28,4.40,4.53
2022-06-14,3.49,4.32,4.44,4.57
2022-06-15,3.52,4.36,4.48,4.61
2022-07-01,3.55,4.05,4.20,4.35
2022-07-04,3.52,4.01,4.16,4.31
2022-07-05,3.50,3.98,4.13,4.28
2022-07-06,3.47,3.94,4.09,4.24
2022-07-07,3.45,3.91,4.06,4.21
2022-07-08,3.42,3.87,4.02,4.17
2022-07-11,3.40,3.84,3.99,4.14
2022-07-12,3.37,3.80,3.95,4.10
2022-07-13,3.35,3.77,3.92,4.07
2022-07-14,3.32,3.73,3.88,4.03
2022-08-01,3.30,3.70,3.85,4.00
2022-08-02,3.27,3.66,3.81,3.96
2022-08-03,3.25,3.63,3.78,3.93
2022-08-04,3.22,3.59,3.74,3.89
2022-08-05,3.20,3.56,3.71,3.86
2022-08-08,3.17,3.52,3.67,3.82
2022-08-09,3.15,3.49,3.64,3.79
2022-08-10,3.12,3.45,3.60,3.75
2022-08-11,3.10,3.42,3.57,3.72
2022-08-12,3.07,3.38,3.53,3.68
2022-09-01,3.50,3.80,3.90,4.02
2022-09-02,3.53,3.84,3.94,4.06
2022-09-05,3.56,3.88,3.98,4.10
2022-09-06,3.59,3.92,4.02,4.14
2022-09-07,3.62,3.96,4.06,4.18
2022-09-08,3.65,4.00,4.10,4.22
2022-09-13,3.70,4.06,4.16,4.28
2022-09-14,3.75,4.12,4.22,4.34
2022-09-15,3.80,4.18,4.28,4.40
2022-09-16,3.85,4.24,4.34,4.46
2022-10-05,4.10,4.50,4.58,4.68
2022-10-06,4.14,4.55,4.63,4.73
2022-10-07,4.18,4.60,4.68,4.78
2022-10-11,4.50,4.90,4.95,5.02
2022-10-12,4.54,4.95,5.00,5.07
2022-10-13,4.58,5.00,5.05,5.12
2022-10-14,4.62,5.05,5.10,5.17
2022-10-17,4.30,4.70,4.78,4.88
2022-10-18,4.26,4.65,4.73,4.83
2022-10-19,4.22,4.60,4.68,4.78
2022-11-01,4.00,4.40,4.50,4.62
2022-11-02,3.97,4.36,4.46,4.58
2022-11-03,3.94,4.32,4.42,4.54
2022-11-04,3.91,4.28,4.38,4.50
2022-11-07,3.88,4.24,4.34,4.46
2022-11-08,3.85,4.20,4.30,4.42
2022-11-09,3.82,4.16,4.26,4.38
2022-11-10,3.79,4.12,4.22,4.34
2022-11-11,3.76,4.08,4.18,4.30
2022-11-14,3.73,4.04,4.14,4.26
2022-12-01,3.60,3.95,4.08,4.22
2022-12-02,3.57,3.91,4.04,4.18
2022-12-05,3.54,3.87,4.00,4.14
2022-12-06,3.51,3.83,3.96,4.10
2022-12-07,3.48,3.79,3.92,4.06
2022-12-08,3.45,3.75,3.88,4.02
2022-12-09,3.42,3.71,3.84,3.98
2022-12-12,3.39,3.67,3.80,3.94
2022-12-13,3.36,3.63,3.76,3.90
2022-12-14,3.33,3.59,3.72,3.86
2023-01-02,3.80,3.70,3.78,3.92
2023-01-03,3.77,3.66,3.74,3.88
2023-01-04,3.74,3.62,3.70,3.84
2023-01-05,3.71,3.58,3.66,3.80
2023-01-06,3.68,3.54,3.62,3.76
2023-01-09,3.65,3.50,3.58,3.72
2023-01-10,3.62,3.46,3.54,3.68
2023-01-11,3.59,3.42,3.50,3.64
2023-01-12,3.56,3.38,3.46,3.60
2023-01-13,3.53,3.34,3.42,3.56
2023-03-01,3.70,3.55,3.60,3.74
2023-03-02,3.68,3.52,3.57,3.71
2023-03-03,3.66,3.49,3.54,3.68
2023-03-06,3.64,3.46,3.51,3.65
2023-03-07,3.62,3.43,3.48,3.62
2023-03-08,3.60,3.40,3.45,3.59
2023-03-09,3.58,3.37,3.42,3.56
2023-03-10,3.56,3.34,3.39,3.53
2023-03-13,3.54,3.31,3.36,3.50
2023-03-14,3.52,3.28,3.33,3.47
2023-06-01,3.80,3.82,3.85,3.94
2023-06-02,3.78,3.79,3.82,3.91
2023-06-05,3.76,3.76,3.79,3.88
2023-06-07,3.74,3.73,3.76,3.85
2023-06-08,3.72,3.70,3.73,3.82
2023-06-09,3.70,3.67,3.70,3.79
2023-09-01,3.95,4.00,4.05,4.15
2023-09-04,3.93,3.97,4.02,4.12
2023-09-05,3.91,3.94,3.99,4.09
2023-09-06,3.89,3.91,3.96,4.06
2023-09-07,3.87,3.88,3.93,4.03
2023-09-08,3.85,3.85,3.90,4.00
2023-11-01,3.75,3.80,3.85,3.95
2023-11-02,3.72,3.76,3.81,3.91
2023-11-03,3.69,3.72,3.77,3.87
2023-11-06,3.66,3.68,3.73,3.83
2023-11-07,3.63,3.64,3.69,3.79
2023-11-08,3.60,3.60,3.65,3.75
2024-01-02,3.55,3.40,3.45,3.58
2024-01-03,3.52,3.36,3.41,3.54
2024-01-04,3.49,3.32,3.37,3.50
2024-01-05,3.46,3.28,3.33,3.46
2024-01-08,3.43,3.24,3.29,3.42
2024-01-09,3.40,3.20,3.25,3.38
2024-01-10,3.37,3.16,3.21,3.34
2024-01-11,3.34,3.12,3.17,3.30
2024-01-12,3.31,3.08,3.13,3.26
2024-01-15,3.28,3.04,3.09,3.22
2024-04-01,3.50,3.30,3.35,3.48
2024-04-02,3.48,3.27,3.32,3.45
2024-04-03,3.46,3.24,3.29,3.42
2024-04-04,3.44,3.21,3.26,3.39
2024-04-05,3.42,3.18,3.23,3.36
2024-04-08,3.40,3.15,3.20,3.33
2024-04-09,3.38,3.12,3.17,3.30
2024-04-10,3.36,3.09,3.14,3.27
2024-04-11,3.34,3.06,3.11,3.24
2024-04-12,3.32,3.03,3.08,3.21
2024-07-01,3.20,2.95,3.02,3.18
2024-07-02,3.18,2.92,2.99,3.15
2024-07-03,3.16,2.89,2.96,3.12
2024-07-04,3.14,2.86,2.93,3.09
2024-07-05,3.12,2.83,2.90,3.06
2024-07-08,3.10,2.80,2.87,3.03
2024-07-09,3.08,2.77,2.84,3.00
2024-07-10,3.06,2.74,2.81,2.97
2024-07-11,3.04,2.71,2.78,2.94
2024-07-12,3.02,2.68,2.75,2.91
2024-10-01,3.10,2.85,2.90,3.05
2024-10-02,3.08,2.82,2.87,3.02
2024-10-04,3.06,2.79,2.84,2.99
2024-10-07,3.04,2.76,2.81,2.96
2024-10-08,3.02,2.73,2.78,2.93
2024-10-10,3.00,2.70,2.75,2.90
2024-10-11,2.98,2.67,2.72,2.87
2024-10-14,2.96,2.64,2.69,2.84
2024-10-15,2.94,2.61,2.66,2.81
2024-10-16,2.92,2.58,2.63,2.78
2025-01-02,2.90,2.55,2.60,2.75
2025-01-03,2.88,2.52,2.57,2.72
2025-01-06,2.86,2.49,2.54,2.69
2025-01-07,2.84,2.46,2.51,2.66
2025-01-08,2.82,2.43,2.48,2.63
2025-01-09,2.80,2.40,2.45,2.60
2025-01-10,2.78,2.37,2.42,2.57
2025-01-13,2.76,2.34,2.39,2.54
2025-01-14,2.74,2.31,2.36,2.51
2025-01-15,2.72,2.28,2.33,2.48
2025-04-01,2.80,2.50,2.55,2.68
2025-04-02,2.78,2.47,2.52,2.65
2025-04-03,2.76,2.44,2.49,2.62
2025-04-04,2.74,2.41,2.46,2.59
2025-04-07,2.72,2.38,2.43,2.56
2025-04-08,2.70,2.35,2.40,2.53
2025-04-09,2.68,2.32,2.37,2.50
2025-04-10,2.66,2.29,2.34,2.47
2025-04-11,2.64,2.26,2.31,2.44
2025-04-12,2.62,2.23,2.28,2.41
"""


def fetch_ktb_yields(api_key: str, start: str, end: str) -> pd.DataFrame:
    """Fetch KTB daily YTM data from ECOS API for all 4 maturities.
    Falls back to embedded sample data when api_key == 'sample'.
    """
    if api_key == 'sample':
        print("[SAMPLE MODE] Using embedded KTB yield data.")
        df = pd.read_csv(StringIO(SAMPLE_CSV), parse_dates=['date'])
        return df.sort_values('date').reset_index(drop=True)

    base = "https://ecos.bok.or.kr/api/StatisticSearch"
    frames = {}
    for label, item_code in MATURITIES.items():
        # END_ROW=2000 covers ~750 trading days (3 years) with room to spare
        url = f"{base}/{api_key}/json/kr/1/2000/{STAT_CODE}/D/{start}/{end}/{item_code}"
        resp = requests.get(url, timeout=15)
        resp.raise_for_status()
        data = resp.json()
        rows = data.get('StatisticSearch', {}).get('row', [])
        if not rows:
            raise ValueError(f"No data returned for {label} (item_code={item_code}). Check API key and dates.")
        s = pd.Series(
            {r['TIME']: float(r['DATA_VALUE']) for r in rows if r['DATA_VALUE']},
            name=f'ytm_{label.lower()}'
        )
        frames[label] = s
        print(f"  {label}: {len(s)} rows fetched")

    df = pd.concat(frames.values(), axis=1)
    df.index = pd.to_datetime(df.index, format='%Y%m%d')
    df.index.name = 'date'
    return df.sort_index().reset_index()


df_raw = fetch_ktb_yields(ECOS_API_KEY, START_DATE, END_DATE)
print(f"\nRows fetched: {len(df_raw)}")
df_raw.head()

## Section 3 — Data Cleaning

In [ ]:
df = df_raw.copy()

# Ensure date column is datetime
df['date'] = pd.to_datetime(df['date'])

# Coerce all yield columns to numeric, replace non-numeric with NaN
ytm_cols = ['ytm_1y', 'ytm_3y', 'ytm_5y', 'ytm_10y']
for col in ytm_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Forward-fill gaps (e.g., public holidays with no data),
# then drop any remaining NaN at the start of the series
df.sort_values('date', inplace=True)
df[ytm_cols] = df[ytm_cols].ffill()
df.dropna(subset=ytm_cols, inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Clean rows: {len(df)}")
print(f"Date range: {df['date'].min().date()} ~ {df['date'].max().date()}")
print(f"\nYTM descriptive statistics (%):\n")
df[ytm_cols].describe().round(4)

## Section 4 — IQR-Based Anomaly Detection

A 60-day rolling window captures ≈ 3 months of market data — long enough to establish  
a stable distributional baseline while remaining responsive to regime changes  
(e.g., BoK rate hiking cycles).  
`min_periods=30` prevents the initial window from producing all-NaN flags.

In [ ]:
WINDOW     = 60
MIN_PERIOD = 30
IQR_MULT   = 1.5

for col in ytm_cols:
    rolling = df[col].rolling(window=WINDOW, min_periods=MIN_PERIOD)
    q1 = rolling.quantile(0.25)
    q3 = rolling.quantile(0.75)
    iqr = q3 - q1
    flag_col = col.replace('ytm_', 'iqr_flag_')
    df[flag_col] = (df[col] < q1 - IQR_MULT * iqr) | (df[col] > q3 + IQR_MULT * iqr)

iqr_flag_cols = [c for c in df.columns if c.startswith('iqr_flag_')]
total_iqr_flags = df[iqr_flag_cols].sum().sum()
print(f"Total IQR anomaly flags: {total_iqr_flags}")
df[iqr_flag_cols].sum().rename('IQR flags per maturity')

## Section 5 — Daily Change Flag (Maturity-Scaled Threshold)

A fixed threshold (e.g., 5bp for all maturities) ignores the natural heteroscedasticity  
of yield changes. Longer-maturity bonds embed a larger **term premium** and **liquidity premium**,  
so we apply maturity-scaled thresholds consistent with Korean bond market convention.

In [ ]:
maturity_map = {'1y': '1Y', '3y': '3Y', '5y': '5Y', '10y': '10Y'}

for col in ytm_cols:
    suffix   = col.replace('ytm_', '')          # '1y', '3y', ...
    mat_key  = maturity_map[suffix]              # '1Y', '3Y', ...
    thresh   = BP_THRESHOLDS[mat_key]            # e.g., 0.05 for 3Y
    daily_ch = df[col].diff().abs()
    bp_col   = col.replace('ytm_', 'bp_flag_')
    df[bp_col] = daily_ch > thresh

bp_flag_cols = [c for c in df.columns if c.startswith('bp_flag_')]
total_bp_flags = df[bp_flag_cols].sum().sum()
print(f"Total ±bp anomaly flags: {total_bp_flags}")
thresholds_display = {k: f"{v*100:.0f}bp" for k, v in BP_THRESHOLDS.items()}
print(f"Thresholds: {thresholds_display}")
df[bp_flag_cols].sum().rename('Daily-change flags per maturity')

## Section 6 — Duration & Convexity

**Modified Duration** is the primary measure of a bond's price sensitivity to yield changes:  
$$\text{MD} = \frac{\text{Macaulay Duration}}{1 + y/m}$$

**Convexity** provides the second-order correction — crucial when yields move by large amounts  
(as seen in Korea's 2022 hiking cycle), where the linear approximation breaks down:

$$\Delta P / P \approx -\text{MD} \cdot \Delta y + \frac{1}{2} \cdot \text{Convexity} \cdot (\Delta y)^2$$

In [ ]:
def modified_duration(ytm: float, coupon_rate: float, periods: int, face: float = 10000.0) -> float:
    """Compute Modified Duration for a semi-annual coupon bond.
    
    Parameters
    ----------
    ytm         : annual YTM as a decimal (e.g., 0.035 for 3.5%)
    coupon_rate : annual coupon rate as a decimal
    periods     : total number of semi-annual coupon periods (e.g., 6 for 3Y)
    face        : face value (KTB standard: 10,000 KRW)
    """
    y = ytm / 2           # semi-annual yield
    c = coupon_rate / 2   # semi-annual coupon rate
    cf = c * face         # semi-annual coupon cash flow

    pv_sum   = 0.0
    wtd_sum  = 0.0
    for t in range(1, periods + 1):
        cashflow = cf if t < periods else cf + face
        pv = cashflow / (1 + y) ** t
        pv_sum  += pv
        wtd_sum += t * pv

    macaulay_duration = wtd_sum / pv_sum / 2   # convert semi-annual periods → years
    return macaulay_duration / (1 + y)


def convexity(ytm: float, coupon_rate: float, periods: int, face: float = 10000.0) -> float:
    """Compute Convexity for a semi-annual coupon bond."""
    y  = ytm / 2
    c  = coupon_rate / 2
    cf = c * face
    price = sum(
        (cf if t < periods else cf + face) / (1 + y) ** t
        for t in range(1, periods + 1)
    )
    convex_sum = sum(
        ((cf if t < periods else cf + face) * t * (t + 1)) / (1 + y) ** (t + 2)
        for t in range(1, periods + 1)
    )
    return convex_sum / price / 4  # divide by m^2=4 to annualize


# Maturity → (coupon_rate, semi-annual periods, label)
BOND_SPECS = {
    'ytm_1y':  (0.025, 2,  '1Y KTB'),
    'ytm_3y':  (0.025, 6,  '3Y KTB'),
    'ytm_5y':  (0.030, 10, '5Y KTB'),
    'ytm_10y': (0.030, 20, '10Y KTB'),
}

for col, (cpn, periods, label) in BOND_SPECS.items():
    df[f'md_{col.split("_")[1]}']  = df[col].apply(
        lambda y: modified_duration(y / 100, cpn, periods) if pd.notna(y) else np.nan
    )
    df[f'cvx_{col.split("_")[1]}'] = df[col].apply(
        lambda y: convexity(y / 100, cpn, periods) if pd.notna(y) else np.nan
    )

md_cols  = [c for c in df.columns if c.startswith('md_')]
cvx_cols = [c for c in df.columns if c.startswith('cvx_')]
print("Modified Duration (years) — latest observation:")
print(df[md_cols].iloc[-1].round(4).to_string())
print("\nConvexity — latest observation:")
print(df[cvx_cols].iloc[-1].round(4).to_string())

## Section 7 — scipy.optimize.brentq YTM Reverse Calculation

We price a **3Y KTB with a fixed 2.5% coupon** (semi-annual, face=10,000).  
The observed YTM on each date is used to compute the theoretical **dirty price**.  
We then recover the YTM from that price using `brentq` — verifying the numerical solver.

**Why this is non-trivial**: when coupon rate ≠ YTM, the bond trades at a discount or premium.  
The YTM equation `PV = Σ CF/(1+y)^t` has no closed-form inverse → numerical root-finding required.

In [ ]:
COUPON_RATE = 0.025   # fixed 2.5% annual coupon (semi-annual payment)
FACE        = 10000.0
PERIODS     = 6       # 3Y × 2 semi-annual periods


def price_from_ytm(ytm: float, face: float, coupon_rate: float, periods: int) -> float:
    """Compute dirty price of a semi-annual coupon bond given YTM (annual, decimal)."""
    y  = ytm / 2
    cf = coupon_rate / 2 * face
    pv = sum(
        cf / (1 + y) ** t for t in range(1, periods + 1)
    )
    pv += face / (1 + y) ** periods
    return pv


def ytm_from_price(price: float, face: float, coupon_rate: float, periods: int) -> float:
    """Recover YTM from bond price using Brent's method.
    Bracket: [1e-4, 0.99] covers 0.01% ~ 99% — safe for all realistic KTB yields.
    """
    objective = lambda ytm: price_from_ytm(ytm, face, coupon_rate, periods) - price
    try:
        return brentq(objective, 1e-4, 0.99, xtol=1e-10)
    except ValueError:
        # f(a) and f(b) have same sign — theoretically impossible in valid range,
        # but guards against NaN or extreme input
        return np.nan


# Apply to each row using observed 3Y YTM
df['theoretical_price'] = df['ytm_3y'].apply(
    lambda y: price_from_ytm(y / 100, FACE, COUPON_RATE, PERIODS) if pd.notna(y) else np.nan
)
df['recovered_ytm'] = df['theoretical_price'].apply(
    lambda p: ytm_from_price(p, FACE, COUPON_RATE, PERIODS) * 100 if pd.notna(p) else np.nan
)
df['ytm_roundtrip_diff_bp'] = (df['recovered_ytm'] - df['ytm_3y']).abs() * 100  # bp

print(f"Coupon rate: {COUPON_RATE*100}% | Face: {FACE:,.0f} KRW | Periods: {PERIODS} semi-annual")
print(f"\nBrentq round-trip error (basis points):")
print(f"  Mean:  {df['ytm_roundtrip_diff_bp'].mean():.6f} bp")
print(f"  Max:   {df['ytm_roundtrip_diff_bp'].max():.6f} bp")
print(f"  All < 0.01bp: {(df['ytm_roundtrip_diff_bp'] < 0.01).all()}")

print("\nSample rows — price vs recovered YTM:")
df[['date', 'ytm_3y', 'theoretical_price', 'recovered_ytm', 'ytm_roundtrip_diff_bp']].head(10).round(4)

## Section 8 — Visualization

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 15))
fig.suptitle('Korean Treasury Bond (국고채) YTM Anomaly Detection\n2022–2025', fontsize=15, fontweight='bold', y=0.98)

colors = {'ytm_1y': '#2196F3', 'ytm_3y': '#4CAF50', 'ytm_5y': '#FF9800', 'ytm_10y': '#9C27B0'}
labels = {'ytm_1y': '1Y', 'ytm_3y': '3Y', 'ytm_5y': '5Y', 'ytm_10y': '10Y'}

# ── Plot 1: YTM Time-Series + IQR Anomaly Markers ────────────────────────────
ax1 = axes[0]
for col in ytm_cols:
    ax1.plot(df['date'], df[col], label=labels[col], color=colors[col], lw=1.5, alpha=0.85)
    flag_col = col.replace('ytm_', 'iqr_flag_')
    anomalies = df[df[flag_col]]
    ax1.scatter(anomalies['date'], anomalies[col], color='red', s=40, zorder=5,
                label='IQR anomaly' if col == 'ytm_1y' else '')

ax1.set_title('Plot 1: KTB YTM Time-Series with IQR Anomalies (60-day rolling window)', fontsize=11)
ax1.set_ylabel('YTM (%)')
ax1.legend(loc='upper right')
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax1.grid(True, alpha=0.3)
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=30)

# ── Plot 2: 3Y Daily Change + Maturity-Scaled Threshold + BP Flags ───────────
ax2 = axes[1]
daily_change_3y = df['ytm_3y'].diff()
thresh_3y = BP_THRESHOLDS['3Y']

ax2.bar(df['date'], daily_change_3y, color='steelblue', alpha=0.6, width=1, label='Daily change (3Y)')
ax2.axhline( thresh_3y, color='red', lw=1.5, ls='--', label=f'+{thresh_3y*100:.0f}bp threshold')
ax2.axhline(-thresh_3y, color='red', lw=1.5, ls='--', label=f'-{thresh_3y*100:.0f}bp threshold')

# Shade ±bp flagged regions
for _, row in df[df['bp_flag_3y']].iterrows():
    ax2.axvspan(row['date'] - pd.Timedelta(days=0.5),
                row['date'] + pd.Timedelta(days=0.5),
                color='red', alpha=0.2)

ax2.set_title('Plot 2: 3Y KTB Daily Yield Change — Maturity-Scaled ±5bp Threshold', fontsize=11)
ax2.set_ylabel('Daily YTM Change (%)')
ax2.legend(loc='upper right')
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax2.grid(True, alpha=0.3)
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=30)

# ── Plot 3: brentq Price vs YTM Scatter (Coupon=2.5%, 3Y KTB) ────────────────
ax3 = axes[2]
sc = ax3.scatter(
    df['ytm_3y'], df['theoretical_price'],
    c=df['date'].apply(lambda d: d.year), cmap='viridis',
    s=20, alpha=0.7
)
plt.colorbar(sc, ax=ax3, label='Year')
ax3.axvline(COUPON_RATE * 100, color='red', ls='--', lw=1.5, label=f'Coupon rate = {COUPON_RATE*100}%')
ax3.axhline(FACE, color='gray', ls=':', lw=1.5, label=f'Par value = {FACE:,.0f}')
ax3.set_title('Plot 3: 3Y KTB Theoretical Price vs Observed YTM\n(Coupon=2.5%, brentq round-trip verified)', fontsize=11)
ax3.set_xlabel('Observed YTM (%)')
ax3.set_ylabel('Theoretical Price (KRW)')
ax3.legend()
ax3.grid(True, alpha=0.3)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig('bond_yield_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print("Chart saved: bond_yield_chart.png")

## Section 9 — Anomaly Summary Report

In [ ]:
# Build combined anomaly log
anomaly_records = []
for col in ytm_cols:
    mat = labels[col]
    iqr_col = col.replace('ytm_', 'iqr_flag_')
    bp_col  = col.replace('ytm_', 'bp_flag_')
    for _, row in df[df[iqr_col] | df[bp_col]].iterrows():
        anomaly_records.append({
            'date':     row['date'].date(),
            'maturity': mat,
            'ytm':      round(row[col], 4),
            'iqr_flag': bool(row[iqr_col]),
            'bp_flag':  bool(row[bp_col]),
            'type':     ('BOTH' if row[iqr_col] and row[bp_col]
                         else 'IQR' if row[iqr_col] else 'BP_CHANGE'),
        })

anomaly_df = pd.DataFrame(anomaly_records).sort_values(['date', 'maturity'])

print("=" * 55)
print("  KTB YTM Anomaly Detection — Summary Report")
print("=" * 55)
print(f"\nTotal unique anomaly events: {len(anomaly_df)}")
print(f"  IQR-only flags:       {(anomaly_df['type']=='IQR').sum()}")
print(f"  Daily-change flags:   {(anomaly_df['type']=='BP_CHANGE').sum()}")
print(f"  Both flags triggered: {(anomaly_df['type']=='BOTH').sum()}")
print()
print("Breakdown by maturity:")
print(anomaly_df.groupby('maturity')['type'].value_counts().to_string())
print()
print("Recent 10 anomaly events:")
anomaly_df.tail(10)